# Train a tank detector

Fine-tune RT-DETR on captured game frames and export a tank-only ONNX model. On Kaggle, attach a zip containing each capture's PNG and COCO JSON, then run all cells with a GPU enabled.

In [ ]:
%pip install -q -U transformers accelerate onnx onnxruntime onnxscript

In [ ]:
import json
import random
import zipfile
from pathlib import Path

import torch
from PIL import Image, ImageDraw

MODEL_ID = 'PekingU/rtdetr_r18vd_coco_o365'
IMG_SIZE = 640
EPOCHS = 40
BATCH_SIZE = 4
SEED = 0
WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./tank_work')
CAPTURE_DIR = WORK / 'captures'
OUTPUT_DIR = WORK / 'tank_detector'
CAPTURE_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)

## Load captures and tank boxes

In [ ]:
for archive in Path('/kaggle/input').rglob('*.zip') if Path('/kaggle/input').is_dir() else []:
    destination = CAPTURE_DIR / archive.stem
    if not destination.exists():
        destination.mkdir(parents=True)
        with zipfile.ZipFile(archive) as zipped:
            zipped.extractall(destination)

json_files = sorted({*Path('/kaggle/input').rglob('*.json'), *Path('captures').rglob('*.json'), *CAPTURE_DIR.rglob('*.json')})
frames = []
for path in json_files:
    try:
        coco = json.loads(path.read_text())
    except (OSError, json.JSONDecodeError):
        continue
    if not isinstance(coco, dict) or 'images' not in coco or 'annotations' not in coco:
        continue
    tank_ids = {c['id'] for c in coco.get('categories', []) if c.get('name', '').lower() == 'tank'}
    if not tank_ids:
        continue
    boxes_by_image = {}
    for annotation in coco['annotations']:
        if annotation.get('category_id') in tank_ids:
            boxes_by_image.setdefault(annotation['image_id'], []).append(annotation['bbox'])
    for image in coco['images']:
        image_path = path.parent / image['file_name']
        if not image_path.is_file():
            image_path = path.with_suffix('.png')
        if image_path.is_file():
            frames.append({'path': image_path, 'id': image['id'], 'boxes': boxes_by_image.get(image['id'], [])})

if not frames:
    raise RuntimeError('No tank captures found. Attach a zip of PNG/COCO JSON captures or add them to captures/.')
random.Random(SEED).shuffle(frames)
val_count = max(1, round(len(frames) * 0.2)) if len(frames) > 1 else 0
val_frames, train_frames = frames[:val_count], frames[val_count:]
if not val_frames:
    val_frames = train_frames
print(f'{len(train_frames)} train / {len(val_frames)} validation frames; {sum(len(f["boxes"]) for f in frames)} tank boxes')

In [ ]:
from torch.utils.data import Dataset
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

processor = RTDetrImageProcessor.from_pretrained(
    MODEL_ID, size={'height': IMG_SIZE, 'width': IMG_SIZE}
)
model = RTDetrForObjectDetection.from_pretrained(
    MODEL_ID, id2label={0: 'tank'}, label2id={'tank': 0}, ignore_mismatched_sizes=True
)

class TankFrames(Dataset):
    def __init__(self, frames):
        self.frames = frames

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, index):
        frame = self.frames[index]
        image = Image.open(frame['path']).convert('RGB')
        annotations = {'image_id': frame['id'], 'annotations': [
            {'image_id': frame['id'], 'category_id': 0, 'bbox': box, 'area': box[2] * box[3], 'iscrowd': 0}
            for box in frame['boxes']
        ]}
        encoded = processor(images=image, annotations=annotations, return_tensors='pt')
        return {'pixel_values': encoded['pixel_values'][0], 'labels': encoded['labels'][0]}

def collate_fn(batch):
    return {'pixel_values': torch.stack([item['pixel_values'] for item in batch]),
            'labels': [item['labels'] for item in batch]}

In [ ]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(OUTPUT_DIR), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=1e-4, weight_decay=1e-4, warmup_ratio=0.05,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        save_total_limit=2, remove_unused_columns=False, fp16=torch.cuda.is_available(),
        report_to='none',
    ),
    train_dataset=TankFrames(train_frames), eval_dataset=TankFrames(val_frames),
    data_collator=collate_fn,
)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))

## Export and verify ONNX

Input is RGB `1×3×640×640` processed by the saved image processor (resize and rescale to 0–1). Output is `1×300×6`: normalized `x1,y1,x2,y2`, tank confidence, and class id (always 0).

In [ ]:
import torch.nn as nn

class BrowserDetector(nn.Module):
    def __init__(self, detector):
        super().__init__()
        self.detector = detector

    def forward(self, pixel_values):
        outputs = self.detector(pixel_values=pixel_values)
        scores = outputs.logits.sigmoid().squeeze(-1)
        center_x, center_y, width, height = outputs.pred_boxes.unbind(-1)
        boxes = torch.stack((center_x - width / 2, center_y - height / 2,
                            center_x + width / 2, center_y + height / 2), dim=-1)
        class_ids = torch.zeros_like(scores)
        return torch.cat((boxes, scores.unsqueeze(-1), class_ids.unsqueeze(-1)), dim=-1)

device = next(model.parameters()).device
detector = BrowserDetector(model).eval()
onnx_path = WORK / 'tank-detector.onnx'
torch.onnx.export(detector, (torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device),), str(onnx_path),
                  input_names=['pixel_values'], output_names=['detections'], opset_version=17)
(WORK / 'tank-detector.labels.txt').write_text('tank\n')

import onnxruntime as ort
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
input_name, output_name = session.get_inputs()[0].name, session.get_outputs()[0].name
test_input = processor(images=Image.open(val_frames[0]['path']).convert('RGB'), return_tensors='pt')['pixel_values'].numpy()
detections = session.run([output_name], {input_name: test_input})[0]
assert detections.shape == (1, 300, 6), detections.shape
print(f'Exported {onnx_path} — verified output {detections.shape}; labels: tank')

Download `tank-detector.onnx` and `tank-detector.labels.txt` from the output and place them in `packages/game/public/models/`. Browser preprocessing must match the saved RT-DETR image processor.